#### MA_filter

In [ ]:
import sys
sys.path.append('../src')

import math
from typing import List

import matplotlib.pyplot as plt

from filter_MA import design_ma, apply_ma

# параметры
fs = 1000          # Гц
T = 1.0            # длительность, с
N_samples = int(fs * T)
t = [n / fs for n in range(N_samples)]

# тестовый сигнал: НЧ + ВЧ
f1 = 5            # низкая частота, Гц
f2 = 100          # высокая частота, Гц

x: List[float] = [
    math.sin(2 * math.pi * f1 * ti) + 0.5 * math.sin(2 * math.pi * f2 * ti)
    for ti in t
]

In [ ]:
# длина окна MA
N_ma = 21

h_ma = design_ma(N_ma)
y_ma = apply_ma(x, h_ma)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(t, x, label="x(t) исходный", alpha=0.7)
plt.plot(t, y_ma, label=f"MA N={N_ma}", alpha=0.7)
plt.xlabel("t, c")
plt.ylabel("Амплитуда")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


### КИХ ВЧ с прямоугольным окном

In [ ]:
from filter_FIR import design_fir_hp_rect, apply_fir

cutoff_hz = 50.0          # срез между 5 Гц и 100 Гц
sample_rate_hz = fs
filter_length = 101       # нечётное, чтобы был чёткий центр

h_hp = design_fir_hp_rect(cutoff_hz, sample_rate_hz, filter_length)
y_hp = apply_fir(x, h_hp)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(t, x, label="x(t) исходный", alpha=0.5)
plt.plot(t, y_hp, label=f"КИХ ВЧ fc={cutoff_hz} Гц", alpha=0.8)
plt.xlabel("t, c")
plt.ylabel("Амплитуда")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

### БИХ ВЧ однополюсный

In [ ]:
from filter_FIR import design_fir_hp_rect, apply_fir
from filter_IIR import design_iir_hp_onepole, apply_iir

fs = 1000.0
fc = 50.0
N_fir = 101

# КИХ ВЧ
h_hp_fir = design_fir_hp_rect(
    cutoff_hz=fc,
    sample_rate_hz=fs,
    filter_length=N_fir,
)
y_hp_fir = apply_fir(x, h_hp_fir)

# Однополюсный ВЧ IIR
a_hp_iir, b_hp_iir = design_iir_hp_onepole(
    cutoff_hz=fc,
    sample_rate_hz=fs,
)
y_hp_iir = apply_iir(x, a_hp_iir, b_hp_iir)
#y_hp_iir = apply_iir(y_hp_iir, a_hp_iir, b_hp_iir)


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(t, x, label="x(t) исходный", alpha=0.4)
plt.plot(t, y_hp_fir, label=f"КИХ ВЧ N={N_fir}", alpha=0.8)
plt.plot(t, y_hp_iir, label="БИХ ВЧ однополюсный", alpha=0.8)
plt.xlabel("t, c")
plt.ylabel("Амплитуда")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import math
from dsp_basic import fft_dit

def pad_to_pow2(z: np.ndarray):
    N = len(z)
    N_fft = 1
    while N_fft < N:
        N_fft *= 2
    z_pad = np.zeros(N_fft, dtype=complex)
    z_pad[:N] = z.astype(complex)
    return z_pad, N, N_fft

# 1. паддинг импульсной характеристики
h = np.array(h_hp_fir, dtype=float)       
h_pad, Nh, Nh_fft = pad_to_pow2(h)

# 2. БПФ
H_fft = fft_dit(list(h_pad))

# 3. амплитуда и ось частот
H_mag = np.array([abs(v) for v in H_fft])

freqs = np.linspace(0, fs, Nh_fft, endpoint=False)  # 0 .. fs-df

# 4. для односторонней АЧХ (0..fs/2)
half = Nh_fft // 2
freqs_half = freqs[:half]
H_mag_half = H_mag[:half]

plt.figure(figsize=(8, 4))
plt.plot(freqs_half, H_mag_half)
plt.xlim(0, 125)
plt.xlabel("f, Гц")
plt.ylabel("|H(f)|")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

from dsp_basic import fft_dit               # твоя реализация FFT
from filter_IIR import design_iir_hp_onepole, apply_iir  # подстрой путь

# параметры фильтра
fs = 1000.0        # частота дискретизации, Гц
fc = 50.0          # частота среза, Гц

# 1. проектируем однополюсный ВЧ БИХ
a_hp, b_hp = design_iir_hp_onepole(
    cutoff_hz=fc,
    sample_rate_hz=fs,
)

# 2. получаем импульсную характеристику (отклик на δ[n])
N_imp = 1024                      # длина импульсной характеристики для АЧХ
x_imp = [0.0] * N_imp
x_imp[0] = 1.0                    # δ[0] = 1, остальные 0

h_iir = apply_iir(x_imp, a_hp, b_hp)   # список длины N_imp

# 3. паддинг до ближайшей степени двойки и FFT
def pad_to_pow2_1d(z):
    arr = np.array(z, dtype=complex)
    N = len(arr)
    N_fft = 1
    while N_fft < N:
        N_fft *= 2
    z_pad = np.zeros(N_fft, dtype=complex)
    z_pad[:N] = arr
    return z_pad, N, N_fft

h_pad, Nh, N_fft = pad_to_pow2_1d(h_iir)

H_fft = fft_dit(list(h_pad))          # список complex
H_mag = np.array([abs(v) for v in H_fft])

# 4. ось частот и перевод в децибелы
freqs = np.linspace(0.0, fs, N_fft, endpoint=False)
half = N_fft // 2
freqs_half = freqs[:half]
H_mag_half = H_mag[:half]

eps = 1e-12
H_db = 20 * np.log10(np.maximum(H_mag_half, eps))

# 5. график АЧХ БИХ ВЧ
plt.figure(figsize=(8, 4))
plt.plot(freqs_half, H_db, label="БИХ ВЧ однополюсный (dB)")
plt.axvline(fc, color="r", linestyle="--", alpha=0.5, label="fc")
plt.xlabel("f, Гц")
plt.ylabel("|H(f)|, dB")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from dsp_basic import fft_dit
from filter_FIR import design_fir_hp_rect
from filter_IIR import design_iir_hp_onepole, apply_iir

# ----- параметры -----
fs = 1000.0       # Гц
fc = 50.0         # Гц
N_fir = 101       # длина КИХ (нечётная)
N_imp_iir = 1024  # длина импульсной характеристики IIR для АЧХ

# ----- КИХ ВЧ: импульсная характеристика -----
h_fir = design_fir_hp_rect(
    cutoff_hz=fc,
    sample_rate_hz=fs,
    filter_length=N_fir,
)                     # список или np.array

# ----- БИХ ВЧ: импульсная характеристика (отклик на delta) -----
a_hp, b_hp = design_iir_hp_onepole(
    cutoff_hz=fc,
    sample_rate_hz=fs,
)

x_imp = [0.0] * N_imp_iir
x_imp[0] = 1.0
h_iir = apply_iir(x_imp, a_hp, b_hp)   # список длины N_imp_iir

# ----- вспомогательная функция паддинга до 2^k -----
def pad_to_pow2_1d(z):
    arr = np.array(z, dtype=complex)
    N = len(arr)
    N_fft = 1
    while N_fft < N:
        N_fft *= 2
    z_pad = np.zeros(N_fft, dtype=complex)
    z_pad[:N] = arr
    return z_pad, N, N_fft

# ----- FFT КИХ -----
h_fir_pad, _, N_fft = pad_to_pow2_1d(h_fir)
H_fir_fft = fft_dit(list(h_fir_pad))
H_fir_mag = np.array([abs(v) for v in H_fir_fft])

# ----- FFT БИХ -----
h_iir_pad, _, N_fft_iir = pad_to_pow2_1d(h_iir)

# чтобы оси совпали, приведём длину FFT для IIR к той же N_fft (при необходимости)
if N_fft_iir != N_fft:
    # простой вариант: перепаддинг до общего максимума
    N_fft_common = max(N_fft, N_fft_iir)
    def repad_to_len(z, N_target):
        arr = np.array(z, dtype=complex)
        out = np.zeros(N_target, dtype=complex)
        out[:len(arr)] = arr[:len(arr)]
        return out

    h_fir_pad = repad_to_len(h_fir_pad, N_fft_common)
    h_iir_pad = repad_to_len(h_iir_pad, N_fft_common)
    N_fft = N_fft_common

    H_fir_fft = fft_dit(list(h_fir_pad))
    H_fir_mag = np.array([abs(v) for v in H_fir_fft])

H_iir_fft = fft_dit(list(h_iir_pad))
H_iir_mag = np.array([abs(v) for v in H_iir_fft])

# ----- ось частот и переход в dB -----
freqs = np.linspace(0.0, fs, N_fft, endpoint=False)
half = N_fft // 2
freqs_half = freqs[:half]

H_fir_half = H_fir_mag[:half]
H_iir_half = H_iir_mag[:half]

eps = 1e-12
H_fir_db = 20 * np.log10(np.maximum(H_fir_half, eps))
H_iir_db = 20 * np.log10(np.maximum(H_iir_half, eps))

# ----- единый график -----
plt.figure(figsize=(9, 4))
plt.plot(freqs_half, H_fir_db, label="КИХ ВЧ (dB)")
plt.plot(freqs_half, H_iir_db, label="БИХ ВЧ однополюсный (dB)")
plt.axvline(fc, color="r", linestyle="--", alpha=0.5, label="fc")
plt.xlabel("f, Гц")
plt.ylabel("|H(f)|, dB")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from dsp_basic import fft_dit
from filter_MA import design_ma
from filter_FIR import design_fir_hp_rect
from filter_IIR import design_iir_hp_onepole, apply_iir

fs = 1000.0
fc = 50.0
N_fir = 101
N_ma = 21
N_imp_iir = 1024

def pad_to_pow2_1d(z):
    arr = np.array(z, dtype=complex)
    N = len(arr)
    N_fft = 1
    while N_fft < N:
        N_fft *= 2
    z_pad = np.zeros(N_fft, dtype=complex)
    z_pad[:N] = arr
    return z_pad, N, N_fft

# --- импульсные характеристики ---
h_ma = design_ma(N_ma)                          # MA
h_fir = design_fir_hp_rect(fc, fs, N_fir)       # КИХ ВЧ

a_hp, b_hp = design_iir_hp_onepole(fc, fs)      # БИХ ВЧ
x_imp = [0.0] * N_imp_iir
x_imp[0] = 1.0
h_iir = apply_iir(x_imp, a_hp, b_hp)

# --- паддинг до общего N_fft ---
h_ma_pad, _, N_fft_ma = pad_to_pow2_1d(h_ma)
h_fir_pad, _, N_fft_fir = pad_to_pow2_1d(h_fir)
h_iir_pad, _, N_fft_iir = pad_to_pow2_1d(h_iir)

N_fft = max(N_fft_ma, N_fft_fir, N_fft_iir)

def repad_to_len(z_pad, N_target):
    arr = np.array(z_pad, dtype=complex)
    out = np.zeros(N_target, dtype=complex)
    out[:len(arr)] = arr[:len(arr)]
    return out

h_ma_pad  = repad_to_len(h_ma_pad,  N_fft)
h_fir_pad = repad_to_len(h_fir_pad, N_fft)
h_iir_pad = repad_to_len(h_iir_pad, N_fft)

# --- FFT и модули ---
H_ma  = fft_dit(list(h_ma_pad))
H_fir = fft_dit(list(h_fir_pad))
H_iir = fft_dit(list(h_iir_pad))

H_ma_mag  = np.abs(H_ma)
H_fir_mag = np.abs(H_fir)
H_iir_mag = np.abs(H_iir)

# --- ось частот и dB ---
freqs = np.linspace(0.0, fs, N_fft, endpoint=False)
half = N_fft // 2
freqs_half = freqs[:half]

eps = 1e-12
H_ma_db  = 20 * np.log10(np.maximum(H_ma_mag[:half],  eps))
H_fir_db = 20 * np.log10(np.maximum(H_fir_mag[:half], eps))
H_iir_db = 20 * np.log10(np.maximum(H_iir_mag[:half], eps))

# --- график ---
plt.figure(figsize=(9, 4))
plt.plot(freqs_half, H_ma_db,  label=f"MA N={N_ma} (НЧ)", alpha=0.8)
plt.plot(freqs_half, H_fir_db, label="КИХ ВЧ", alpha=0.8)
plt.plot(freqs_half, H_iir_db, label="БИХ ВЧ однополюсный", alpha=0.8)
plt.axvline(fc, color="r", linestyle="--", alpha=0.5, label="fc")
plt.xlabel("f, Гц")
plt.ylabel("|H(f)|, dB")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()
